# 🛡️ Python Encapsulation & Properties — The Complete Guide

> **Module:** Object-Oriented Programming (OOP) | **Notebook 4 of 5**

Encapsulation is the fundamental OOP principle of bundling data and the methods that operate on that data inside a single unit, while restricting direct external access to internal state to protect data integrity and prevent unintended side effects.

Python approaches encapsulation with a unique and pragmatic philosophy: *"We are all consenting adults here"*. Rather than rigid compiler-enforced access barriers (`private`, `protected`, `public`), Python utilizes clear naming conventions, the elegant `@property` decorator, custom reusable **Descriptors**, and modern immutable **Dataclasses**.

---

## 📋 Table of Contents
1. [Introduction to Encapsulation & Python's Privacy Model](#1.-Introduction-to-Encapsulation-&-Python's-Privacy-Model)
2. [Access Modifiers & Name Mangling Mechanics](#2.-Access-Modifiers-&-Name-Mangling-Mechanics)
3. [The `@property` Decorator: Getters, Setters, & Deleters](#3.-The-@property-Decorator:-Getters,-Setters,-&-Deleters)
4. [Custom Descriptors Protocol (`__get__`, `__set__`, `__set_name__`)](#4.-Custom-Descriptors-Protocol-(__get__,-__set__,-__set_name__))
5. [Performance Optimization with `@functools.cached_property`](#5.-Performance-Optimization-with-@functools.cached_property)
6. [Immutable Encapsulation with Frozen Dataclasses](#6.-Immutable-Encapsulation-with-Frozen-Dataclasses)
7. [Real-World Case Studies & Architectural Patterns](#7.-Real-World-Case-Studies-&-Architectural-Patterns)
   - 7.1 [Secure User Account & Password Vault](#7.1-Secure-User-Account-&-Password-Vault)
   - 7.2 [Mini-ORM Data Model Validation Framework](#7.2-Mini-ORM-Data-Model-Validation-Framework)
   - 7.3 [Multidirectional Temperature & Climate Telemetry](#7.3-Multidirectional-Temperature-&-Climate-Telemetry)
8. [Common Pitfalls & Anti-Patterns](#8.-Common-Pitfalls-&-Anti-Patterns)
9. [Hands-On Interactive Challenges](#9.-Hands-On-Interactive-Challenges)
10. [Quick Reference Card & Summary](#10.-Quick-Reference-Card-&-Summary)


---
## 1. Introduction to Encapsulation & Python's Privacy Model

### 🔐 What is Encapsulation?
Encapsulation serves two primary purposes:
1. **Data Bundling**: Keeping related attributes and methods organized inside cohesive class structures.
2. **Information Hiding & Invariant Protection**: Protecting internal state against unauthorized direct mutation, ensuring that the object always remains in a valid, consistent state.

### 🐍 Python's Privacy Conventions

| Access Level | Naming Convention | Meaning / Intent | Enforcement Mechanism |
| :--- | :--- | :--- | :--- |
| **Public** | `self.name` | Part of the class public API; unrestricted access | None (Open access) |
| **Protected** | `self._balance` | Internal implementation detail; intended for class & subclasses | PEP 8 convention (Not enforced by compiler) |
| **Private** | `self.__secret` | Strict internal private attribute | **Name Mangling** (`_ClassName__secret`) |


In [ ]:
class AccountPrivacyDemo:
    def __init__(self, owner: str, balance: float, pin: int):
        self.owner = owner        # Public attribute
        self._balance = balance   # Protected attribute (Convention)
        self.__pin = pin          # Private attribute (Name Mangled)

    def verify_pin(self, entered_pin: int) -> bool:
        """Controlled internal access method."""
        return self.__pin == entered_pin

demo = AccountPrivacyDemo("Alice", 1500.0, 4821)

print("1. Public access:    ", demo.owner)
print("2. Protected access: ", demo._balance, " (Accessible, but violates convention!)")

# 3. Direct private access raises AttributeError:
try:
    print(demo.__pin)
except AttributeError as e:
    print("3. Private access:    AttributeError ->", e)


---
## 2. Access Modifiers & Name Mangling Mechanics

### ⚙️ How Name Mangling Works Under the Hood
When Python encounters an identifier starting with at least two leading underscores (and at most one trailing underscore, e.g. `__pin`), it textually transforms it to:
`_ClassName__attribute`

### 🎯 Why Does Name Mangling Exist?
Name Mangling was **not** designed for security or data encryption. 
Its true purpose is to **prevent accidental attribute name collisions in subclass hierarchies**!


In [ ]:
class BaseService:
    def __init__(self):
        self.__config = {"timeout": 30}  # Mangled to _BaseService__config

    def get_config(self):
        return self.__config

class CustomService(BaseService):
    def __init__(self):
        super().__init__()
        # Subclass defines its own __config without accidentally overwriting the parent's!
        self.__config = {"retries": 3}   # Mangled to _CustomService__config

svc = CustomService()

# Inspecting instance namespace:
print("Instance __dict__ keys:")
for k, v in svc.__dict__.items():
    print(f"  {k} = {v}")

print(f"\nBaseService config:   {svc.get_config()}")
print(f"CustomService config: {svc._CustomService__config}")


---
## 3. The `@property` Decorator: Getters, Setters, & Deleters

### 🪄 The Pythonic Way to Encapsulate Data
In Java or C++, developers write manual getter and setter methods (`get_balance()`, `set_balance()`).
In Python, this is considered an anti-pattern. Instead, start with a simple public attribute, and if validation or computation is later needed, convert it into a **`@property`** without breaking existing client code!

- `@property`: Defines the **getter**.
- `@<attr>.setter`: Defines the **setter** (runs on `obj.attr = new_val`).
- `@<attr>.deleter`: Defines the **deleter** (runs on `del obj.attr`).


In [ ]:
class BankAccount:
    """Encapsulated bank account using properties for state validation."""

    def __init__(self, account_holder: str, initial_balance: float = 0.0):
        self.account_holder = account_holder
        # Use property setter for initial validation
        self.balance = initial_balance

    # 1. Getter for balance
    @property
    def balance(self) -> float:
        """The current account balance in USD."""
        return self._balance

    # 2. Setter with validation rules
    @balance.setter
    def balance(self, new_balance: float) -> None:
        if not isinstance(new_balance, (int, float)):
            raise TypeError("Balance must be a numeric value.")
        if new_balance < 0:
            raise ValueError(f"Balance cannot be negative; received ${new_balance:,.2f}")
        self._balance = float(new_balance)

    # 3. Computed / Derived Property (Read-Only)
    @property
    def is_overdrawn(self) -> bool:
        return self._balance <= 0

account = BankAccount("Ada Lovelace", 2500.0)

# Natural attribute access syntax triggers getter & setter functions:
print(f"Current balance: ${account.balance:,.2f}")
print(f"Is overdrawn?    {account.is_overdrawn}")

account.balance += 500.0  # Runs getter, adds 500, runs setter
print(f"Updated balance: ${account.balance:,.2f}")

# Validation prevents invalid state:
try:
    account.balance = -100.0
except ValueError as e:
    print(f"\n[Validation Error Caught] {e}")


---
## 4. Custom Descriptors Protocol (`__get__`, `__set__`, `__set_name__`)

### 🧠 What is a Descriptor?
If you have 10 attributes across multiple classes that all require positive number validation, writing 10 `@property` getters and setters causes massive code duplication (DRY violation).

A **Descriptor** is an object attribute whose access behavior is overridden by methods in the descriptor protocol:
- `__get__(self, instance, owner)`: Triggered on attribute read.
- `__set__(self, instance, value)`: Triggered on attribute write.
- `__delete__(self, instance)`: Triggered on attribute deletion.
- `__set_name__(self, owner, name)`: (Python 3.6+) Automatically captures the attribute name assigned in the class!


In [ ]:
class PositiveFloat:
    """Reusable validation descriptor for positive floating point attributes."""
    
    def __set_name__(self, owner, name):
        # Automatically capture the attribute name (e.g. 'price' or 'weight')
        self.private_name = f"_{name}"

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return getattr(instance, self.private_name, 0.0)

    def __set__(self, instance, value):
        if not isinstance(value, (int, float)):
            raise TypeError(f"Attribute '{self.private_name[1:]}' must be numeric.")
        if value <= 0:
            raise ValueError(f"Attribute '{self.private_name[1:]}' must be strictly positive (> 0); received {value}")
        setattr(instance, self.private_name, float(value))

# Applying the descriptor across multiple attributes seamlessly:
class ProductItem:
    # Class-level descriptors
    price = PositiveFloat()
    weight_kg = PositiveFloat()
    shipping_cost = PositiveFloat()

    def __init__(self, name: str, price: float, weight_kg: float, shipping_cost: float):
        self.name = name
        self.price = price
        self.weight_kg = weight_kg
        self.shipping_cost = shipping_cost

item = ProductItem("Ergonomic Keyboard", price=129.99, weight_kg=1.2, shipping_cost=15.0)

print(f"Product: {item.name} | Price: ${item.price} | Weight: {item.weight_kg}kg | Shipping: ${item.shipping_cost}")

# Descriptor validation works automatically:
try:
    item.price = -50.0
except ValueError as e:
    print(f"\n[Descriptor Validation Caught] {e}")


---
## 5. Performance Optimization with `@functools.cached_property`

### ⚡ `@property` vs. `@cached_property`
- Standard `@property`: Executed **every single time** the attribute is accessed.
- `@functools.cached_property` (Python 3.8+): Executed on the **first access**, then cached directly into the instance's `__dict__`. Subsequent accesses are direct dictionary lookups ($O(1)$) with zero overhead!
- Cache invalidation: simply `del instance.attribute` to force recomputation on next read.


In [ ]:
from functools import cached_property
import time

class DatasetAnalyzer:
    def __init__(self, data_points: list[float]):
        self.data_points = data_points

    @cached_property
    def statistical_summary(self) -> dict[str, float]:
        """Simulates an expensive data processing computation."""
        print("  [CALCULATING] Running heavy statistics calculation...")
        time.sleep(0.05)  # Simulate CPU delay
        
        n = len(self.data_points)
        mean_val = sum(self.data_points) / n
        variance = sum((x - mean_val) ** 2 for x in self.data_points) / n
        return {
            "count": n,
            "mean": round(mean_val, 2),
            "std_dev": round(variance ** 0.5, 2),
            "min": min(self.data_points),
            "max": max(self.data_points)
        }

analyzer = DatasetAnalyzer([12.5, 45.2, 78.9, 23.4, 99.1, 54.3, 88.7])

print("First access (Triggers calculation):")
res1 = analyzer.statistical_summary
print("Result 1:", res1)

print("\nSecond access (Instant cache hit):")
res2 = analyzer.statistical_summary
print("Result 2:", res2)

print("\nInvalidating cache via 'del analyzer.statistical_summary':")
del analyzer.statistical_summary

print("Third access after cache invalidation (Recalculates):")
res3 = analyzer.statistical_summary
print("Result 3:", res3)


---
## 6. Immutable Encapsulation with Frozen Dataclasses

### 🧊 Creating Immutable Value Objects
Using `@dataclass(frozen=True)` provides total encapsulation:
- All fields become read-only after initialization.
- Python automatically generates `__repr__`, `__eq__`, and `__hash__` methods.
- Instances can be safely used as dictionary keys or stored in `set` collections.


In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class GeoCoordinate:
    """Immutable geographic coordinate value object."""
    latitude: float
    longitude: float
    city_name: str

    def __post_init__(self):
        # Validation inside frozen dataclass
        if not (-90.0 <= self.latitude <= 90.0):
            raise ValueError(f"Latitude {self.latitude} must be in [-90, 90]")
        if not (-180.0 <= self.longitude <= 180.0):
            raise ValueError(f"Longitude {self.longitude} must be in [-180, 180]")

loc1 = GeoCoordinate(37.7749, -122.4194, "San Francisco")
print(f"Created location: {loc1}")

# Attempting mutation raises FrozenInstanceError:
try:
    loc1.latitude = 40.7128
except Exception as e:
    print(f"\n[Immutability Protected] Cannot mutate frozen dataclass: {type(e).__name__}")

# Safe to use as dict keys or in sets:
locations_set = {loc1, GeoCoordinate(51.5074, -0.1278, "London")}
print(f"Total locations in set: {len(locations_set)}")


---
## 7. Real-World Case Studies & Architectural Patterns

---

### 7.1 Secure User Account & Password Vault
Demonstrating hashed password setters, masked getters, and token rotation.


In [ ]:
import hashlib
import os

class SecureUserVault:
    """Demonstrates write-only properties and masked privacy getters."""

    def __init__(self, username: str, raw_password: str, ssn: str):
        self.username = username
        self._salt = os.urandom(16).hex()
        self.password = raw_password  # Uses property setter for hashing
        self._ssn = ssn

    @property
    def password(self):
        """Write-only property: prevents reading plain or hashed passwords."""
        raise PermissionError("Password is write-only for security reasons.")

    @password.setter
    def password(self, new_raw_password: str) -> None:
        if len(new_raw_password) < 8:
            raise ValueError("Password must be at least 8 characters long.")
        # Hash with salt using SHA-256
        salted = (new_raw_password + self._salt).encode("utf-8")
        self._password_hash = hashlib.sha256(salted).hexdigest()

    def verify_password(self, candidate_password: str) -> bool:
        salted = (candidate_password + self._salt).encode("utf-8")
        return hashlib.sha256(salted).hexdigest() == self._password_hash

    @property
    def masked_ssn(self) -> str:
        """Masked getter: displays only last 4 digits (e.g. ***-**-1234)."""
        clean_ssn = self._ssn.replace("-", "").strip()
        return f"***-**-{clean_ssn[-4:]}"

user = SecureUserVault("grace_hopper", "Adm1ral_Grace#1906", "123-45-6789")

print(f"User: {user.username} | Masked SSN: {user.masked_ssn}")
print("Verify correct password:  ", user.verify_password("Adm1ral_Grace#1906"))
print("Verify incorrect password:", user.verify_password("wrong_pass"))

try:
    _ = user.password
except PermissionError as e:
    print(f"[Security Check] {e}")


---
### 7.2 Mini-ORM Data Model Validation Framework
Building an extensible ORM-like model where attributes are validated via reusable descriptors.


In [ ]:
class StringField:
    def __init__(self, min_len: int = 1, max_len: int = 255):
        self.min_len = min_len
        self.max_len = max_len

    def __set_name__(self, owner, name):
        self.field_name = name
        self.storage_name = f"_{name}"

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return getattr(instance, self.storage_name, None)

    def __set__(self, instance, value):
        if not isinstance(value, str):
            raise TypeError(f"Field '{self.field_name}' must be a string.")
        if not (self.min_len <= len(value) <= self.max_len):
            raise ValueError(f"Field '{self.field_name}' length must be between {self.min_len} and {self.max_len}.")
        setattr(instance, self.storage_name, value.strip())

class IntegerField:
    def __init__(self, min_val: int = 0, max_val: int = 1_000_000):
        self.min_val = min_val
        self.max_val = max_val

    def __set_name__(self, owner, name):
        self.field_name = name
        self.storage_name = f"_{name}"

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return getattr(instance, self.storage_name, None)

    def __set__(self, instance, value):
        if not isinstance(value, int):
            raise TypeError(f"Field '{self.field_name}' must be an integer.")
        if not (self.min_val <= value <= self.max_val):
            raise ValueError(f"Field '{self.field_name}' must be between {self.min_val} and {self.max_val}.")
        setattr(instance, self.storage_name, value)

class CustomerRecord:
    first_name = StringField(min_len=2, max_len=50)
    last_name = StringField(min_len=2, max_len=50)
    age = IntegerField(min_val=18, max_val=120)
    loyalty_points = IntegerField(min_val=0, max_val=100000)

    def __init__(self, first_name: str, last_name: str, age: int, loyalty_points: int = 0):
        self.first_name = first_name
        self.last_name = last_name
        self.age = age
        self.loyalty_points = loyalty_points

# Valid Customer Creation
cust = CustomerRecord("Alan", "Turing", age=41, loyalty_points=5000)
print(f"Customer: {cust.first_name} {cust.last_name} | Age: {cust.age} | Points: {cust.loyalty_points}")

# Validation failure
try:
    cust.age = 15  # Under 18
except ValueError as e:
    print(f"[ORM Validation Error] {e}")


---
### 7.3 Multidirectional Temperature & Climate Telemetry
A class where `celsius`, `fahrenheit`, and `kelvin` are all interchangeable `@property` interfaces pointing to a single normalized internal Kelvin storage.


In [ ]:
class ClimateSensor:
    """Maintains single source of truth (Kelvin) while exposing 3 temperature units."""

    def __init__(self, temp_celsius: float = 0.0):
        self.celsius = temp_celsius  # Uses setter

    # 1. Celsius Interface
    @property
    def celsius(self) -> float:
        return round(self._kelvin - 273.15, 2)

    @celsius.setter
    def celsius(self, value: float) -> None:
        self.kelvin = value + 273.15  # Delegate to Kelvin setter for absolute zero check

    # 2. Fahrenheit Interface
    @property
    def fahrenheit(self) -> float:
        return round((self.celsius * 9 / 5) + 32, 2)

    @fahrenheit.setter
    def fahrenheit(self, value: float) -> None:
        self.celsius = (value - 32) * 5 / 9

    # 3. Normalized Kelvin Internal Interface
    @property
    def kelvin(self) -> float:
        return round(self._kelvin, 2)

    @kelvin.setter
    def kelvin(self, value: float) -> None:
        if value < 0:
            raise ValueError(f"Temperature {value}K is below Absolute Zero (0K)!")
        self._kelvin = float(value)

sensor = ClimateSensor(25.0)  # 25 C Room Temperature

print(f"Reading: {sensor.celsius} C == {sensor.fahrenheit} F == {sensor.kelvin} K")

sensor.fahrenheit = 212.0     # Boiling point of water
print(f"After setting 212 F -> {sensor.celsius} C == {sensor.kelvin} K")

try:
    sensor.celsius = -300.0   # Below Absolute Zero (-273.15 C)
except ValueError as e:
    print(f"[Physics Violation Prevented] {e}")


---
## 8. Common Pitfalls & Anti-Patterns

### ❌ Pitfall 1: Infinite Recursion Inside `@property.setter`
Writing `self.balance = new_balance` inside `def balance(self, new_balance):` recursively calls the setter until Python crashes with `RecursionError`! Always store in `self._balance`.

### ❌ Pitfall 2: Premature Over-Encapsulation
Writing getters and setters for every single attribute from day one without any validation or computation needed. In Python, start with public attributes. You can always convert to `@property` later without breaking API callers!

### ❌ Pitfall 3: Assuming Private Attributes (`__secret`) are Encrypted
Name mangling is merely a namespace transformation (`_Class__secret`). Anyone can still read or modify it directly if they want. Never store raw unencrypted secrets in memory assuming Name Mangling provides security.


---
## 9. Hands-On Interactive Challenges

---

### 🎯 Challenge 1: Validated Bank Account Property
Create a `SecureAccount` class with:
- Read-only property `account_number` (set once during `__init__`).
- Property `balance` that only accepts numeric values $\ge 0$.
- Property `interest_rate` that only accepts values between `0.0` and `0.20` (0% to 20%).


In [ ]:
class SecureAccount:
    def __init__(self, account_number: str, initial_balance: float = 0.0, interest_rate: float = 0.03):
        self._account_number = str(account_number).strip()
        self.balance = initial_balance
        self.interest_rate = interest_rate

    @property
    def account_number(self) -> str:
        return self._account_number

    @property
    def balance(self) -> float:
        return self._balance

    @balance.setter
    def balance(self, value: float) -> None:
        if not isinstance(value, (int, float)):
            raise TypeError("Balance must be numeric.")
        if value < 0:
            raise ValueError("Balance cannot be negative.")
        self._balance = float(value)

    @property
    def interest_rate(self) -> float:
        return self._interest_rate

    @interest_rate.setter
    def interest_rate(self, value: float) -> None:
        if not (0.0 <= value <= 0.20):
            raise ValueError("Interest rate must be between 0.0 and 0.20.")
        self._interest_rate = float(value)

# Automated verification test
acc = SecureAccount("ACC-901", 1000.0, 0.05)
assert acc.account_number == "ACC-901"
assert acc.balance == 1000.0
assert acc.interest_rate == 0.05

acc.balance += 250.0
assert acc.balance == 1250.0

try:
    acc.account_number = "ACC-999"  # Read only!
    assert False, "Should raise AttributeError"
except AttributeError:
    pass

try:
    acc.interest_rate = 0.50  # Over 20%
    assert False, "Should raise ValueError"
except ValueError:
    pass

print("[OK] Challenge 1 Passed!")


---
### 🎯 Challenge 2: Reusable Bounded Range Descriptor
Implement a descriptor `BoundedRange(min_val, max_val)` that validates numerical bounds on any assigned attribute using `__set_name__`.


In [ ]:
class BoundedRange:
    def __init__(self, min_val: float, max_val: float):
        self.min_val = min_val
        self.max_val = max_val

    def __set_name__(self, owner, name):
        self.storage_name = f"_{name}"
        self.field_name = name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return getattr(instance, self.storage_name, self.min_val)

    def __set__(self, instance, value):
        if not isinstance(value, (int, float)):
            raise TypeError(f"'{self.field_name}' must be a number.")
        if not (self.min_val <= value <= self.max_val):
            raise ValueError(f"'{self.field_name}' must be between {self.min_val} and {self.max_val}; got {value}")
        setattr(instance, self.storage_name, float(value))

class AudioMixer:
    volume = BoundedRange(0.0, 100.0)
    pan = BoundedRange(-1.0, 1.0)

    def __init__(self, volume: float = 50.0, pan: float = 0.0):
        self.volume = volume
        self.pan = pan

# Automated verification test
mixer = AudioMixer(75.0, 0.5)
assert mixer.volume == 75.0
assert mixer.pan == 0.5

mixer.volume = 100.0
mixer.pan = -1.0

try:
    mixer.volume = 105.0  # Out of bounds
    assert False, "Should raise ValueError"
except ValueError:
    pass

print("[OK] Challenge 2 Passed!")


---
### 🎯 Challenge 3: Immutable Bounding Box with Computed Properties
Implement a `@dataclass(frozen=True)` `BoundingBox` with:
- Fields: `x: float`, `y: float`, `width: float`, `height: float`.
- Computed properties: `area` ($width \times height$), `center` ($(x + \frac{w}{2}, y + \frac{h}{2})$).
- Method `contains(px: float, py: float) -> bool`.


In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class BoundingBox:
    x: float
    y: float
    width: float
    height: float

    def __post_init__(self):
        if self.width <= 0 or self.height <= 0:
            raise ValueError("Width and height must be strictly positive.")

    @property
    def area(self) -> float:
        return self.width * self.height

    @property
    def center(self) -> tuple[float, float]:
        return (self.x + self.width / 2.0, self.y + self.height / 2.0)

    def contains(self, px: float, py: float) -> bool:
        return (self.x <= px <= self.x + self.width) and (self.y <= py <= self.y + self.height)

# Automated verification test
bbox = BoundingBox(10.0, 20.0, 100.0, 50.0)

assert bbox.area == 5000.0
assert bbox.center == (60.0, 45.0)
assert bbox.contains(50.0, 30.0) is True
assert bbox.contains(5.0, 30.0) is False

try:
    bbox.width = 200.0  # Frozen instance!
    assert False, "Should raise FrozenInstanceError"
except Exception:
    pass

print("[OK] Challenge 3 Passed!")


---
## 10. Quick Reference Card & Summary

### 💡 Complete Encapsulation Reference Cell


In [ ]:
# ============================================================
# PYTHON ENCAPSULATION & PROPERTIES — QUICK REFERENCE
# ============================================================

from functools import cached_property

class QuickRefEncapsulation:
    def __init__(self, name: str, salary: float, secret_pin: int):
        self.name = name            # 1. Public
        self._salary = salary       # 2. Protected (Convention)
        self.__secret_pin = secret_pin # 3. Private (Name Mangled)

    # 4. Property Getter
    @property
    def salary(self) -> float:
        return self._salary

    # 5. Property Setter
    @salary.setter
    def salary(self, value: float) -> None:
        if value < 0:
            raise ValueError("Salary cannot be negative.")
        self._salary = value

    # 6. Read-Only Computed Property
    @property
    def annual_salary(self) -> float:
        return self._salary * 12

    # 7. Cached Property (evaluated once)
    @cached_property
    def tax_estimate(self) -> float:
        return self.annual_salary * 0.25

# Demo execution
emp = QuickRefEncapsulation("Alan Turing", 8000.0, 1234)
emp.salary = 8500.0

print(f"Monthly: ${emp.salary:,.2f} | Annual: ${emp.annual_salary:,.2f} | Est Tax: ${emp.tax_estimate:,.2f}")
print(f"Mangled PIN attribute: {emp._QuickRefEncapsulation__secret_pin}")


### 📊 Master Encapsulation Cheat Sheet

| Mechanism | Syntax | Best Use Case |
| :--- | :--- | :--- |
| **Public Attribute** | `self.x = val` | Default for general attributes requiring no validation |
| **Protected Attribute** | `self._x = val` | Internal state intended for class and subclass use only |
| **Private (Name Mangling)**| `self.__x = val` | Avoiding attribute collisions in complex inheritance hierarchies |
| **Property Getter** | `@property def x(self):` | Exposing managed, validated, or computed read access |
| **Property Setter** | `@x.setter def x(self, v):` | Enforcing validation rules, type checks, and side effects |
| **Cached Property** | `@cached_property` | Expensive computations computed once and cached on instance |
| **Descriptor** | `class Desc: def __set_name__` | DRY, reusable validation logic across multiple classes |
| **Frozen Dataclass** | `@dataclass(frozen=True)` | Immutable, hashable value objects with zero boilerplate |

---
*Next up in OOP Series → **Magic (Dunder) Methods (`dunder_methods.ipynb`)***
